In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import json

from sklearn.metrics import mean_absolute_error, f1_score

Mounted at /content/drive


In [2]:
import tensorflow as tf
print("Using GPU:", tf.test.is_gpu_available())

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Using GPU: True


In [13]:
DATA_DIR = "/content/drive/MyDrive/FYP/phase3"

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]

train_houses = list(range(1, 17))   # 1–16
test_houses  = list(range(17, 21))  # 17–20

In [24]:
def load_data(appliance, houses, max_samples=5000):
    X_all, y_all = [], []

    for file in os.listdir(DATA_DIR):
        if appliance not in file or not file.endswith(".npz"):
            continue

        house_id = int(file.split("_")[0].replace("house", ""))
        if house_id not in houses:
            continue

        path = os.path.join(DATA_DIR, file)
        data = np.load(path)

        print(f"Loading: {file}")

        X = data["X"]
        y = data["y"]

        X = X[:max_samples]
        y = y[:max_samples]

        X_all.append(X)
        y_all.append(y)

    return np.vstack(X_all), np.hstack(y_all)

In [25]:
def detect_scale(y):
    max_val = np.max(y)

    if max_val <= 2:
        return "normalized"
    elif max_val <= 50:
        return "small"
    else:
        return "watts"

In [26]:
def compute_threshold_and_epsilon(y_train):
    scale = detect_scale(y_train)

    # ---------- EPSILON ----------
    if scale == "normalized":
        epsilon = 0.05
    elif scale == "small":
        epsilon = max(0.5, np.percentile(y_train, 20))
    else:
        epsilon = max(10, np.percentile(y_train, 20))

    # ---------- THRESHOLD ----------
    y_on = y_train[y_train > epsilon]

    if len(y_on) < 10:
        threshold = np.percentile(y_train, 80)
    else:
        threshold = np.percentile(y_on, 40)

    return threshold, epsilon, scale

In [30]:
def evaluate_threshold(X_test, y_test, threshold, epsilon, scale):
    X_test_mean = np.mean(X_test, axis=1)

    # ---------- SCALE MATCH ----------
    if scale == "normalized":
        scale_factor = 1.0
    else:
        y_valid = y_test[y_test > epsilon]

        if len(y_valid) < 10:
            scale_factor = 0.1
        else:
            scale_factor = np.median(y_valid) / (np.median(X_test_mean) + 1e-6)

    y_pred_power = X_test_mean * scale_factor
    y_pred_power = np.nan_to_num(y_pred_power)

    # ---------- CLASSIFICATION ----------
    y_true = (y_test > epsilon).astype(int)
    # Add tolerance band
    margin = threshold * 0.2

    y_pred = (y_pred_power > (threshold - margin)).astype(int)

    # ---------- METRICS ----------
    mae = mean_absolute_error(y_test, y_pred_power)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return mae, f1, y_true, y_pred

In [31]:
results = {}

for app in APPLIANCES:
    print("\n==============================")
    print("Processing:", app)
    print("==============================")

    X_train, y_train = load_data(app, train_houses)
    X_test, y_test   = load_data(app, test_houses)

    print("Train:", X_train.shape, "Test:", X_test.shape)

    threshold, epsilon, scale = compute_threshold_and_epsilon(y_train)

    print("Detected scale:", scale)
    print("Epsilon:", epsilon)
    print("Threshold:", threshold)

    mae, f1, y_true, y_pred = evaluate_threshold(
        X_test, y_test, threshold, epsilon, scale
    )

    print("MAE:", mae)
    print("F1 Score:", f1)
    print("True ON ratio:", np.mean(y_true))
    print("Pred ON ratio:", np.mean(y_pred))

    results[app] = {
        "MAE": mae,
        "F1": f1,
        "threshold": threshold,
        "epsilon": epsilon,
        "scale": scale
    }

    # Save for Raspberry Pi
    with open(f"/content/{app}_threshold.json", "w") as f:
        json.dump({
            "threshold": float(threshold),
            "epsilon": float(epsilon)
        }, f)


Processing: toaster
Loading: house2_toaster_seq2point.npz
Loading: house3_toaster_seq2point.npz
Loading: house5_toaster_seq2point.npz
Loading: house6_toaster_seq2point.npz
Loading: house7_toaster_seq2point.npz
Loading: house8_toaster_seq2point.npz
Loading: house10_toaster_seq2point.npz
Loading: house12_toaster_seq2point.npz
Loading: house14_toaster_seq2point.npz
Loading: house18_toaster_seq2point.npz
Train: (45000, 599) Test: (5000, 599)
Detected scale: normalized
Epsilon: 0.05
Threshold: 0.4285
MAE: 0.06806869804859161
F1 Score: 0.0
True ON ratio: 0.0022
Pred ON ratio: 0.0

Processing: kettle
Loading: house2_kettle_seq2point.npz
Loading: house3_kettle_seq2point.npz
Loading: house4_kettle_seq2point.npz
Loading: house5_kettle_seq2point.npz
Loading: house6_kettle_seq2point.npz
Loading: house7_kettle_seq2point.npz
Loading: house8_kettle_seq2point.npz
Loading: house9_kettle_seq2point.npz
Loading: house11_kettle_seq2point.npz
Loading: house12_kettle_seq2point.npz
Loading: house13_kettle_se

In [35]:
from google.colab import files

for app in APPLIANCES:
    files.download(f"/content/{app}_threshold.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
print("\n========== FINAL THRESHOLD RESULTS ==========")

for app, res in results.items():
    print(f"\n{app.upper()}")
    print(f"Scale     : {res['scale']}")
    print(f"MAE       : {res['MAE']:.4f}")
    print(f"F1        : {res['F1']:.4f}")
    print(f"Threshold : {res['threshold']:.4f}")
    print(f"Epsilon   : {res['epsilon']:.4f}")


========== FINAL THRESHOLD RESULTS ==========

TOASTER
Scale     : normalized
MAE       : 0.0681
F1        : 0.0000
Threshold : 0.4285
Epsilon   : 0.0500

KETTLE
Scale     : normalized
MAE       : 0.0708
F1        : 0.0000
Threshold : 0.7129
Epsilon   : 0.0500

COMPUTER
Scale     : normalized
MAE       : 0.0732
F1        : 0.0314
Threshold : 0.1480
Epsilon   : 0.0500

LAMP
Scale     : small
MAE       : 0.5971
F1        : 0.1252
Threshold : 0.5600
Epsilon   : 0.5000
